# Phase 4 — Ablation
## Brain Tumour MRI Classification
====================================================================

Which parts of the recipe actually do something, measured one factor at a time
against a noise floor established from repeated runs of the same baseline.

The test set is not touched here either. Every number is validation.

In [1]:
import sys, time
from pathlib import Path
sys.path.insert(0, str(Path.cwd().parent))

import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import numpy as np

from src import config, engine, manifest, metrics, splits, viz
from src.config import CACHE_DIR, CLASSES, DEVICE, IMG_SIZE, TEST_DIR, TRAIN_DIR
from src.model import receptive_field

train_img, train_lab, _, test_img, _, _, C = \
    splits.build_caches(TRAIN_DIR, TEST_DIR)
MEAN, STD = np.load(CACHE_DIR / "norm.npy")
S = splits.build_splits(train_img, train_lab, test_img)
train_idx, val_idx = S["train_idx"], S["val_idx"]
assert manifest.read()["split_hash"] == S["split_hash"], "re-run Phase 1"

print(f"device {DEVICE}   full train pool {len(train_idx)}   val {len(val_idx)}")

device cuda   full train pool 4224   val 704


In [2]:
# 1. WHAT AN ABLATION IS FOR, AND HOW THIS ONE WAS WRONG BEFORE
"""
An ablation answers one question: does this component earn its place? Not "is
the final model good", which the test set answers, but "would removing this
change anything measurable".

The trap is that neural network training is noisy, so any single comparison
between two configurations is partly measuring the seed. The defence is a noise
floor: run the same baseline several times with different seeds, see how much
the answer moves on its own, and treat differences smaller than that as nothing.

A previous version of this study got the defence wrong in a way worth recording,
because the failure is easy to repeat. It used three seeds and took max minus min
as the threshold. Three samples give a terrible estimate of spread, and a range
is a worse statistic than a standard deviation because it can only grow as you
add samples. The resulting bar was 0.0154 -- wider than nearly every real effect
-- so the study reported label smoothing and augmentation as noise. A later
controlled run on the full training split showed the configuration it had
dismissed beating the one it endorsed by 0.0115, comfortably outside the seed
spread.

So this version uses five seeds and reports the standard error of the mean, and
judges a factor by whether its effect exceeds two standard deviations of the
baseline. The distinction being drawn is between "we measured no effect" and
"we could not have detected one", which are not the same statement.
"""
print("factors under test:")
for f in ("deeper (receptive field 38px -> 62px)", "label smoothing 0.1",
          "dropout 0.4", "weight decay 1e-4", "augmentation (affine + jitter)",
          "corruption augmentation (blur + noise)", "input resolution 192",
          "activation ELU"):
    print(f"  - {f}")

factors under test:
  - deeper (receptive field 38px -> 62px)
  - label smoothing 0.1
  - dropout 0.4
  - weight decay 1e-4
  - augmentation (affine + jitter)
  - corruption augmentation (blur + noise)
  - input resolution 192
  - activation ELU


In [3]:
# 2. THE PROTOCOL
"""
Three decisions define this study and each trades something away.

A stratified subset instead of the full training split. Every configuration
below would otherwise take fifteen minutes, and there are more than a dozen. The
cost is that absolute accuracies here are lower than Phase 3's -- these numbers
compare configurations to each other, and are not results.

A fixed epoch budget with no early stopping, so no configuration is advantaged
by being allowed to train longer. This has a known side effect: a regulariser
that slows convergence is penalised by a short budget, which is one reason
augmentation can look harmful on a small subset and help on the full set.

One factor at a time from a fixed baseline rather than a grid. A full grid over
eight factors is 256 runs. The limitation is real and worth stating plainly:
this design cannot detect interactions, so if two techniques only help together,
this study will report that neither helps.

The baseline is deliberately unregularised and shallow, so every factor has room
to show an effect.
"""
SUB_PER_CLASS, ABL_EPOCHS, SEEDS = 250, 25, (42, 7, 1234, 2024, 99)
rng = np.random.default_rng(config.SEED)
sub_idx = train_idx[np.concatenate(
    [rng.choice(np.where(train_lab[train_idx] == c)[0], SUB_PER_CLASS, replace=False)
     for c in range(len(CLASSES))])]

BASELINE = dict(dropout=0.0, weight_decay=0.0, augment=False, corrupt=False,
                label_smoothing=0.0, deep=False, activation="relu",
                img_size=IMG_SIZE, balance=None)

print(f"training subset  {len(sub_idx)} images ({SUB_PER_CLASS}/class)")
print(f"validation       {len(val_idx)} images (the full validation split)")
print(f"epochs           {ABL_EPOCHS}, no early stopping")
print(f"noise floor      {len(SEEDS)} seeds\n")
print("baseline configuration:")
for k, v in BASELINE.items():
    print(f"  {k:<18} {v}")

def run(tag, seed=config.SEED, **overrides):
    cfg = {**BASELINE, **overrides}
    t0 = time.time()
    h = engine.run_experiment(train_img, train_lab, sub_idx, val_idx, MEAN, STD,
                              epochs=ABL_EPOCHS, seed=seed, **cfg)
    y, p, _ = metrics.predict(h["model"], h["val_loader"])
    f1 = metrics.macro_f1(y, p, len(CLASSES))
    s = engine.summarise(h)
    print(f"  {tag:<30} macro F1 {f1:.4f}   val acc {s['best_val_acc']:.4f}"
          f"   ({time.time()-t0:.0f}s)")
    return {"tag": tag, "f1": f1, "acc": s["best_val_acc"]}

results = []

training subset  1000 images (250/class)
validation       704 images (the full validation split)
epochs           25, no early stopping
noise floor      5 seeds

baseline configuration:
  dropout            0.0
  weight_decay       0.0
  augment            False
  corrupt            False
  label_smoothing    0.0
  deep               False
  activation         relu
  img_size           128
  balance            None


In [4]:
# 3. THE NOISE FLOOR
"""
The same configuration, five times, differing only in seed. Whatever spread
appears here is the amount by which a number can move for no reason at all, and
nothing smaller than it counts as a finding.

Reported as a standard deviation and a standard error, not a range. The range of
five samples is a worse estimator than the standard deviation of five samples,
and it systematically grows as you add runs, which makes it useless for
comparing studies of different sizes.
"""
seed_f1 = []
for seed in SEEDS:
    r = run(f"baseline seed {seed}", seed=seed)
    seed_f1.append(r["f1"])

seed_f1 = np.array(seed_f1)
BASE_F1 = float(seed_f1.mean())
NOISE   = float(seed_f1.std(ddof=1))
SEM     = NOISE / np.sqrt(len(SEEDS))
THRESH  = 2 * NOISE

print(f"\nbaseline macro F1 over {len(SEEDS)} seeds")
print(f"  mean            {BASE_F1:.4f}")
print(f"  std dev         {NOISE:.4f}")
print(f"  standard error  {SEM:.4f}")
print(f"  range           {seed_f1.min():.4f} - {seed_f1.max():.4f}")
print(f"\n-> a factor counts as signal only if it moves macro F1 by more than")
print(f"   two standard deviations, {THRESH:.4f}")

results.append({"tag": "baseline (none)", "f1": BASE_F1, "acc": float('nan')})

  baseline seed 42               macro F1 0.8769   val acc 0.8778   (51s)


  baseline seed 7                macro F1 0.8868   val acc 0.8821   (52s)


  baseline seed 1234             macro F1 0.8791   val acc 0.8892   (52s)


  baseline seed 2024             macro F1 0.8879   val acc 0.8892   (52s)


  baseline seed 99               macro F1 0.8830   val acc 0.8892   (62s)

baseline macro F1 over 5 seeds
  mean            0.8827
  std dev         0.0048
  standard error  0.0021
  range           0.8769 - 0.8879

-> a factor counts as signal only if it moves macro F1 by more than
   two standard deviations, 0.0095


In [5]:
# 4. ONE FACTOR AT A TIME
"""
Each run changes exactly one thing from the baseline. The receptive-field
prediction recorded in Phase 2 is the one to watch: the deeper variant should
win, and should win because 38px of a 128px scan is too little context to judge
a lesion's margin, not because it carries more parameters.

ELU is here because it was proposed as an improvement. Proposing is not
measuring, and the honest way to settle it is a row in this table. Its Kaiming
gain is switched to the linear approximation, since reusing the ReLU gain would
make this a test of a badly scaled initialisation instead of the activation.
"""
for tag, kw in [
    ("deeper (RF 38->62px)",       dict(deep=True)),
    ("label smoothing 0.1",        dict(label_smoothing=0.1)),
    ("dropout 0.4",                dict(dropout=0.4)),
    ("weight decay 1e-4",          dict(weight_decay=1e-4)),
    ("augment (affine+jitter)",    dict(augment=True)),
    ("augment (affine only)",      dict(augment=True, jitter=False)),
    ("corruption aug (blur+noise)", dict(augment=True, corrupt=True)),
    ("resolution 192",             dict(img_size=192)),
    ("activation ELU",             dict(activation="elu")),
]:
    results.append(run(tag, **kw))

  deeper (RF 38->62px)           macro F1 0.8950   val acc 0.8963   (129s)


  label smoothing 0.1            macro F1 0.8866   val acc 0.8949   (106s)


  dropout 0.4                    macro F1 0.8712   val acc 0.8707   (107s)


  weight decay 1e-4              macro F1 0.8729   val acc 0.8736   (107s)


  augment (affine+jitter)        macro F1 0.8721   val acc 0.8722   (111s)


  augment (affine only)          macro F1 0.8775   val acc 0.8764   (115s)


  corruption aug (blur+noise)    macro F1 0.8344   val acc 0.8352   (136s)


  resolution 192                 macro F1 0.8658   val acc 0.8665   (270s)


  activation ELU                 macro F1 0.8640   val acc 0.8679   (114s)


In [6]:
# 5. THE RESULTS TABLE
"""
Sorted by effect. The signal column is the only one that should be read as a
conclusion, and it says one of three things: signal, noise, or -- for anything
between one and two standard deviations -- undetermined at this sample size.

That third category is the honest one and the previous study did not have it.
Collapsing "no effect" and "an effect too small for this design to see" into a
single verdict is what produced its wrong answer.
"""
ranked = sorted(results, key=lambda r: -r["f1"])
print(f"{'configuration':<30}{'macro F1':>10}{'vs base':>10}{'signal':>14}")
print("-" * 64)
for r in ranked:
    d = r["f1"] - BASE_F1
    if r["tag"].startswith("baseline"):
        verdict = "-"
    elif abs(d) > THRESH:
        verdict = "yes" if d > 0 else "yes (hurts)"
    elif abs(d) > NOISE:
        verdict = "undetermined"
    else:
        verdict = "noise"
    print(f"{r['tag']:<30}{r['f1']:>10.4f}{d:>+10.4f}{verdict:>14}")

fig, ax = viz.styled_fig(figsize=(9, 4.5))
tags = [r["tag"] for r in ranked]
deltas = [r["f1"] - BASE_F1 for r in ranked]
cols = [config.PALETTE["good"] if d > THRESH else
        config.PALETTE["val"] if d < -THRESH else config.PALETTE["train"]
        for d in deltas]
ax.barh(range(len(tags)), deltas, color=cols)
ax.axvline(0, color='k', lw=1)
for x, ls in ((THRESH, '--'), (-THRESH, '--')):
    ax.axvline(x, color='k', ls=ls, lw=1, alpha=0.5)
ax.set_yticks(range(len(tags))); ax.set_yticklabels(tags, fontsize=8)
ax.invert_yaxis(); ax.set_xlabel("macro F1 change vs baseline")
ax.set_facecolor(config.FACE)
ax.set_title(f"One factor at a time (dashed = 2 sd noise floor, {THRESH:.4f})",
             fontsize=11, fontweight='bold')
plt.tight_layout(); viz.save(fig, "ablation.png")

configuration                   macro F1   vs base        signal
----------------------------------------------------------------
deeper (RF 38->62px)              0.8950   +0.0123           yes
label smoothing 0.1               0.8866   +0.0039         noise
baseline (none)                   0.8827   +0.0000             -
augment (affine only)             0.8775   -0.0053  undetermined
weight decay 1e-4                 0.8729   -0.0098   yes (hurts)
augment (affine+jitter)           0.8721   -0.0106   yes (hurts)
dropout 0.4                       0.8712   -0.0115   yes (hurts)
resolution 192                    0.8658   -0.0169   yes (hurts)
activation ELU                    0.8640   -0.0187   yes (hurts)
corruption aug (blur+noise)       0.8344   -0.0484   yes (hurts)
  saved -> outputs/ablation.png


WindowsPath('C:/Games/Codes/Python/Projects/Brain_Tumour_Detection/Final_Project/outputs/ablation.png')

In [7]:
# 6. CONFIRMING ON THE FULL TRAINING SPLIT
"""
The subset study is where this design is weakest, and the previous version's
error came from trusting it too far. A factor that fails on 1000 images and 25
epochs has not been shown to fail on 4224 images and 60 -- regularisers in
particular need data and time before they pay.

So the factors that cleared the bar are combined and trained once on the full
split, against the same validation set Phase 3 used, which makes the comparison
against the shipped model direct.
"""
winners = [r for r in ranked
           if not r["tag"].startswith("baseline") and r["f1"] - BASE_F1 > THRESH]
combined = dict(BASELINE)
for r in winners:
    tag = r["tag"]
    if tag.startswith("deeper"):        combined.update(deep=True)
    elif tag.startswith("label"):       combined.update(label_smoothing=0.1)
    elif tag.startswith("dropout"):     combined.update(dropout=0.4)
    elif tag.startswith("weight"):      combined.update(weight_decay=1e-4)
    elif tag.startswith("augment"):     combined.update(augment=True,
                                                        jitter="jitter" in tag)
    elif tag.startswith("corruption"):  combined.update(augment=True, corrupt=True)
    elif tag.startswith("resolution"):  combined.update(img_size=192)
    elif tag.startswith("activation"):  combined.update(activation="elu")

print(f"factors clearing the {THRESH:.4f} bar: {[r['tag'] for r in winners] or 'none'}")
print("\ncombined configuration:")
for k, v in combined.items():
    print(f"  {k:<18} {v}")

t0 = time.time()
h_full = engine.run_experiment(train_img, train_lab, train_idx, val_idx, MEAN, STD,
                               epochs=30, verbose=True, **combined)
y, p, _ = metrics.predict(h_full["model"], h_full["val_loader"])
F1_FULL = metrics.macro_f1(y, p, len(CLASSES))
print(f"\nfull-data combined: macro F1 {F1_FULL:.4f}   "
      f"val acc {engine.summarise(h_full)['best_val_acc']:.4f}   ({time.time()-t0:.0f}s)")

h03 = np.load(config.OUTPUTS / "history.npy", allow_pickle=True).item()
print(f"Phase 3 shipped model:  val acc {max(h03['val_acc']):.4f} "
      f"(full data, {h03['stopped_at']} epochs, early stopping)")

factors clearing the 0.0095 bar: ['deeper (RF 38->62px)']

combined configuration:
  dropout            0.0
  weight_decay       0.0
  augment            False
  corrupt            False
  label_smoothing    0.0
  deep               True
  activation         relu
  img_size           128
  balance            None


  epoch   1/30  train 0.8125/0.6586   val 0.7351/0.6790  <- best


  epoch   5/30  train 0.3513/0.8705   val 1.4103/0.6108


  epoch  10/30  train 0.1996/0.9280   val 0.4582/0.8267


  epoch  15/30  train 0.1078/0.9616   val 0.3146/0.9176


  epoch  20/30  train 0.0309/0.9882   val 0.2197/0.9403


  epoch  25/30  train 0.0110/0.9962   val 0.1914/0.9560


  epoch  30/30  train 0.0038/0.9988   val 0.1938/0.9588



full-data combined: macro F1 0.9585   val acc 0.9616   (801s)
Phase 3 shipped model:  val acc 0.9759 (full data, 60 epochs, early stopping)


In [8]:
# 7. SUMMARY
"""
What this study supports and what it does not, stated separately, because the
difference is where ablations usually overclaim.
"""
print("=" * 66)
print("PHASE 4 — ABLATION SUMMARY")
print("=" * 66)
print(f"  baseline ({len(SEEDS)} seeds)   {BASE_F1:.4f}  sd {NOISE:.4f}  sem {SEM:.4f}")
print(f"  signal threshold      {THRESH:.4f}  (2 sd)")
print(f"  best single factor    {ranked[0]['tag']} ({ranked[0]['f1']:.4f})")
print(f"  factors above bar     {len(winners)} of {len(results)-1}")
print(f"  combined, full data   {F1_FULL:.4f} macro F1")

rf_row = next((r for r in results if r["tag"].startswith("deeper")), None)
print(f"""
  THE PHASE 2 PREDICTION

  Phase 2 recorded, before any of this was run, that the deeper variant should
  win and should win on receptive field rather than parameter count: 38px is
  30% of a 128px scan, too little to judge a lesion's margin against the tissue
  around it. Measured here: {rf_row['f1'] - BASE_F1:+.4f} macro F1 against a
  {THRESH:.4f} bar.

  WHAT THIS SUPPORTS

  Each factor was measured against a fixed baseline under an identical budget,
  with a noise floor from {len(SEEDS)} repeated runs. Differences reported as
  signal exceed two standard deviations of that floor.

  WHAT IT DOES NOT

  One factor at a time cannot detect interactions. If two techniques only help
  in combination, this design reports that neither helps.

  The sweep ran on {len(sub_idx)} of {len(train_idx)} training images for
  {ABL_EPOCHS} epochs, so absolute numbers are not comparable to Phase 3 and a
  fixed short budget penalises any regulariser that slows convergence.

  Factors marked "undetermined" are not factors shown to do nothing. They are
  factors this design could not resolve, which is a statement about the study
  rather than about them.

  Every number here is validation. Phase 5 opens the test set.""")

PHASE 4 — ABLATION SUMMARY
  baseline (5 seeds)   0.8827  sd 0.0048  sem 0.0021
  signal threshold      0.0095  (2 sd)
  best single factor    deeper (RF 38->62px) (0.8950)
  factors above bar     1 of 9
  combined, full data   0.9585 macro F1

  THE PHASE 2 PREDICTION

  Phase 2 recorded, before any of this was run, that the deeper variant should
  win and should win on receptive field rather than parameter count: 38px is
  30% of a 128px scan, too little to judge a lesion's margin against the tissue
  around it. Measured here: +0.0123 macro F1 against a
  0.0095 bar.

  WHAT THIS SUPPORTS

  Each factor was measured against a fixed baseline under an identical budget,
  with a noise floor from 5 repeated runs. Differences reported as
  signal exceed two standard deviations of that floor.

  WHAT IT DOES NOT

  One factor at a time cannot detect interactions. If two techniques only help
  in combination, this design reports that neither helps.

  The sweep ran on 1000 of 4224 training 